**TEXT-RANK**


In [1]:
import re
from pprint import pprint

import numpy as np
from nltk import sent_tokenize, word_tokenize

from nltk.cluster.util import cosine_distance

MULTIPLE_WHITESPACE_PATTERN = re.compile(r"\s+", re.UNICODE)


def normalize_whitespace(text):
    """
    Translates multiple whitespace into single space character.
    If there is at least one new line character chunk is replaced
    by single LF (Unix new line) character.
    """
    return MULTIPLE_WHITESPACE_PATTERN.sub(_replace_whitespace, text)


def _replace_whitespace(match):
    text = match.group()

    if "\n" in text or "\r" in text:
        return "\n"
    else:
        return " "


def is_blank(string):
    """
    Returns `True` if string contains only white-space characters
    or is empty. Otherwise `False` is returned.
    """
    return not string or string.isspace()


def get_symmetric_matrix(matrix):
    """
    Get Symmetric matrix
    :param matrix:
    :return: matrix
    """
    return matrix + matrix.T - np.diag(matrix.diagonal())


def core_cosine_similarity(vector1, vector2):
    """
    measure cosine similarity between two vectors
    :param vector1:
    :param vector2:
    :return: 0 < cosine similarity value < 1
    """
    return 1 - cosine_distance(vector1, vector2)


'''
Note: This is not a summarization algorithm. This Algorithm pics top sentences irrespective of the order they appeared.
'''


class TextRank4Sentences():
    def __init__(self):
        self.damping = 0.85  # damping coefficient, usually is .85
        self.min_diff = 1e-5  # convergence threshold
        self.steps = 100  # iteration steps
        self.text_str = None
        self.sentences = None
        self.pr_vector = None

    def _sentence_similarity(self, sent1, sent2, stopwords=None):
        if stopwords is None:
            stopwords = []

        sent1 = [w.lower() for w in sent1]
        sent2 = [w.lower() for w in sent2]

        all_words = list(set(sent1 + sent2))

        vector1 = [0] * len(all_words)
        vector2 = [0] * len(all_words)

        # build the vector for the first sentence
        for w in sent1:
            if w in stopwords:
                continue
            vector1[all_words.index(w)] += 1

        # build the vector for the second sentence
        for w in sent2:
            if w in stopwords:
                continue
            vector2[all_words.index(w)] += 1

        return core_cosine_similarity(vector1, vector2)

    def _build_similarity_matrix(self, sentences, stopwords=None):
        # create an empty similarity matrix
        sm = np.zeros([len(sentences), len(sentences)])

        for idx1 in range(len(sentences)):
            for idx2 in range(len(sentences)):
                if idx1 == idx2:
                    continue

                sm[idx1][idx2] = self._sentence_similarity(sentences[idx1], sentences[idx2], stopwords=stopwords)

        # Get Symmeric matrix
        sm = get_symmetric_matrix(sm)

        # Normalize matrix by column
        norm = np.sum(sm, axis=0)
        sm_norm = np.divide(sm, norm, where=norm != 0)  # this is to ignore the 0 element in norm

        return sm_norm

    def _run_page_rank(self, similarity_matrix):

        pr_vector = np.array([1] * len(similarity_matrix))

        # Iteration
        previous_pr = 0
        for epoch in range(self.steps):
            pr_vector = (1 - self.damping) + self.damping * np.matmul(similarity_matrix, pr_vector)
            if abs(previous_pr - sum(pr_vector)) < self.min_diff:
                break
            else:
                previous_pr = sum(pr_vector)

        return pr_vector

    def _get_sentence(self, index):

        try:
            return self.sentences[index]
        except IndexError:
            return ""

    def get_top_sentences(self, number=5):

        top_sentences = {}

        if self.pr_vector is not None:

            sorted_pr = np.argsort(self.pr_vector)
            sorted_pr = list(sorted_pr)
            sorted_pr.reverse()

            index = 0
            for epoch in range(number):
                print (str(sorted_pr[index]) + " : " + str(self.pr_vector[sorted_pr[index]]))
                sent = self.sentences[sorted_pr[index]]
                sent = normalize_whitespace(sent)
                top_sentences[sent] = self.pr_vector[sorted_pr[index]]
                index += 1

        return top_sentences

    def analyze(self, text, stop_words=None):
        self.text_str = text
        self.sentences = sent_tokenize(self.text_str)

        tokenized_sentences = [word_tokenize(sent) for sent in self.sentences]

        similarity_matrix = self._build_similarity_matrix(tokenized_sentences, stop_words)

        self.pr_vector = self._run_page_rank(similarity_matrix)
        print(self.pr_vector)


text_str = '''
    Those Who Are Resilient Stay In The Game Longer
    “On the mountains of truth you can never climb in vain: either you will reach a point higher up today, or you will be training your powers so that you will be able to climb higher tomorrow.” — Friedrich Nietzsche
    Challenges and setbacks are not meant to defeat you, but promote you. However, I realise after many years of defeats, it can crush your spirit and it is easier to give up than risk further setbacks and disappointments. Have you experienced this before? To be honest, I don’t have the answers. I can’t tell you what the right course of action is; only you will know. However, it’s important not to be discouraged by failure when pursuing a goal or a dream, since failure itself means different things to different people. To a person with a Fixed Mindset failure is a blow to their self-esteem, yet to a person with a Growth Mindset, it’s an opportunity to improve and find new ways to overcome their obstacles. Same failure, yet different responses. Who is right and who is wrong? Neither. Each person has a different mindset that decides their outcome. Those who are resilient stay in the game longer and draw on their inner means to succeed.
    '''



[1.27124764 1.09149529 0.49907963 1.24689674 1.08644157 1.24013595
 1.24151475 0.9964572  0.6098623  0.81096533 0.8501204  1.05578319]
0 : 1.271247636683798
3 : 1.2468967412932273
6 : 1.2415147486210913
5 : 1.2401359537858996
1 : 1.0914952900297563
{'\nThose Who Are Resilient Stay In The Game Longer\n“On the mountains of truth you can never climb in vain: either you will reach a point higher up today, or you will be training your powers so that you will be able to climb higher tomorrow.” — Friedrich Nietzsche\nChallenges and setbacks are not meant to defeat you, but promote you.': 1.271247636683798,
 'However, I realise after many years of defeats, it can crush your spirit and it is easier to give up than risk further setbacks and disappointments.': 1.0914952900297563,
 'However, it’s important not to be discouraged by failure when pursuing a goal or a dream, since failure itself means different things to different people.': 1.2401359537858996,
 'To a person with a Fixed Mindset failur

In [2]:
text="I There are houses in certain provincial towns whose aspect inspires melancholy, akin to that called forth by sombre cloisters, dreary moorlands, or the desolation of ruins. Within these houses there is, perhaps, the silence of the cloister, the barrenness of moors, the skeleton of ruins; life and movement are so stagnant there that a stranger might think them uninhabited, were it not that he encounters suddenly the pale, cold glance of a motionless person, whose half-monastic face peers beyond the window-casing at the sound of an unaccustomed step. Such elements of sadness formed the physiognomy, as it were, of a dwelling-house in Saumur which stands at the end of the steep street leading to the chateau in the upper part of the town. This street--now little frequented, hot in summer, cold in winter, dark in certain sections--is remarkable for the resonance of its little pebbly pavement, always clean and dry, for the narrowness of its tortuous road-way, for the peaceful stillness of its houses, which belong to the Old town and are over-topped by the ramparts. Houses three centuries old are still solid, though built of wood, and their divers aspects add to the originality which commends this portion of Saumur to the attention of artists and antiquaries. It is difficult to pass these houses without admiring the enormous oaken beams, their ends carved into fantastic figures, which crown with a black bas-relief the lower floor of most of them. In one place these transverse timbers are covered with slate and mark a bluish line along the frail wall of a dwelling covered by a roof _en colombage_ which bends beneath the weight of years, and whose rotting shingles are twisted by the alternate action of sun and rain. In another place blackened, worn-out window-sills, with delicate sculptures now scarcely discernible, seem too weak to bear the brown clay pots from which springs the heart’s-ease or the rose-bush of some poor working-woman. Farther on are doors studded with enormous nails, where the genius of our forefathers has traced domestic hieroglyphics, of which the meaning is now lost forever. Here a Protestant attested his belief; there a Leaguer cursed Henry IV.; elsewhere some bourgeois has carved the insignia of his _noblesse de cloches_, symbols of his long-forgotten magisterial glory. The whole history of France is there. Next to a tottering house with roughly plastered walls, where an artisan enshrines his tools, rises the mansion of a country gentleman, on the stone arch of which above the door vestiges of armorial bearings may still be seen, battered by the many revolutions that have shaken France since 1789. In this hilly street the ground-floors of the merchants are neither shops nor warehouses; lovers of the Middle Ages will here find the _ouvrouere_ of our forefathers in all its naive simplicity. These low rooms, which have no shop-frontage, no show-windows, in fact no glass at all, are deep and dark and without interior or exterior decoration. Their doors open in two parts, each roughly iron-bound; the upper half is fastened back within the room, the lower half, fitted with a spring-bell, swings continually to and fro. Air and light reach the damp den within, either through the upper half of the door, or through an open space between the ceiling and a low front wall, breast-high, which is closed by solid shutters that are taken down every morning, put up every evening, and held in place by heavy iron bars. This wall serves as a counter for the merchandise. No delusive display is there; only samples of the business, whatever it may chance to be,--such, for instance, as three or four tubs full of codfish and salt, a few bundles of sail-cloth, cordage, copper wire hanging from the joists above, iron hoops for casks ranged along the wall, or a few pieces of cloth upon the shelves. Enter. A neat girl, glowing with youth, wearing a white kerchief, her arms red and bare, drops her knitting and calls her father or her mother, one of whom comes forward and sells you what you want, phlegmatically, civilly, or arrogantly, according to his or her individual character, whether it be a matter of two sous’ or twenty thousand francs’ worth of merchandise. You may see a cooper, for instance, sitting in his doorway and twirling his thumbs as he talks with a neighbor. To all appearance he owns nothing more than a few miserable boat-ribs and two or three bundles of laths; but below in the port his teeming wood-yard supplies all the cooperage trade of Anjou. He knows to a plank how many casks are needed if the vintage is good. A hot season makes him rich, a rainy season ruins him; in a single morning puncheons worth eleven francs have been known to drop to six. In this country, as in Touraine, atmospheric vicissitudes control commercial life. Wine-growers, proprietors, wood-merchants, coopers, inn-keepers, mariners, all keep watch of the sun. They tremble when they go to bed lest they should hear in the morning of a frost in the night; they dread rain, wind, drought, and want water, heat, and clouds to suit their fancy. A perpetual duel goes on between the heavens and their terrestrial interests. The barometer smooths, saddens, or makes merry their countenances, turn and turn about. From end to end of this street, formerly the Grand’Rue de Saumur, the words: “Here’s golden weather,” are passed from door to door; or each man calls to his neighbor: “It rains louis,” knowing well what a sunbeam or the opportune rainfall is bringing him. On Saturdays after midday, in the fine season, not one sou’s worth of merchandise can be bought from these worthy traders. Each has his vineyard, his enclosure of fields, and all spend two days in the country. This being foreseen, and purchases, sales, and profits provided for, the merchants have ten or twelve hours to spend in parties of pleasure, in making observations, in criticisms, and in continual spying. A housewife cannot buy a partridge without the neighbors asking the husband if it were cooked to a turn. A young girl never puts her head near a window that she is not seen by idling groups in the street. Consciences are held in the light; and the houses, dark, silent, impenetrable as they seem, hide no mysteries. Life is almost wholly in the open air; every household sits at its own threshold, breakfasts, dines, and quarrels there. No one can pass along the street without being examined; in fact formerly, when a stranger entered a provincial town he was bantered and made game of from door to door. From this came many good stories, and the nickname _copieux_, which was applied to the inhabitants of Angers, who excelled in such urban sarcasms. The ancient mansions of the old town of Saumur are at the top of this hilly street, and were formerly occupied by the nobility of the neighborhood. The melancholy dwelling where the events of the following history took place is one of these mansions,--venerable relics of a century in which men and things bore the characteristics of simplicity which French manners and customs are losing day by day. Follow the windings of the picturesque thoroughfare, whose irregularities awaken recollections that plunge the mind mechanically into reverie, and you will see a somewhat dark recess, in the centre of which is hidden the door of the house of Monsieur Grandet. It is impossible to understand the force of this provincial expression--the house of Monsieur Grandet--without giving the biography of Monsieur Grandet himself. Monsieur Grandet enjoyed a reputation in Saumur whose causes and effects can never be fully understood by those who have not, at one time or another, lived in the provinces. In 1789 Monsieur Grandet--still called by certain persons le Pere Grandet, though the number of such old persons has perceptibly diminished--was a master-cooper, able to read, write, and cipher. At the period when the French Republic offered for sale the church property in the arrondissement of Saumur, the cooper, then forty years of age, had just married the daughter of a rich wood-merchant. Supplied with the ready money of his own fortune and his wife’s _dot_, in all about two thousand louis-d’or, Grandet went to the newly established “district,” where, with the help of two hundred double louis given by his father-in-law to the surly republican who presided over the sales of the national domain, he obtained for a song, legally if not legitimately, one of the finest vineyards in the arrondissement, an old abbey, and several farms. The inhabitants of Saumur were so little revolutionary that they thought Pere Grandet a bold man, a republican, and a patriot with a mind open to all the new ideas; though in point of fact it was open only to vineyards. He was appointed a member of the administration of Saumur, and his pacific influence made itself felt politically and commercially. Politically, he protected the ci-devant nobles, and prevented, to the extent of his power, the sale of the lands and property of the _emigres_; commercially, he furnished the Republican armies with two or three thousand puncheons of white wine, and took his pay in splendid fields belonging to a community of women whose lands had been reserved for the last lot. Under the Consulate Grandet became mayor, governed wisely, and harvested still better pickings. Under the Empire he was called Monsieur Grandet. Napoleon, however, did not like republicans, and superseded Monsieur Grandet (who was supposed to have worn the Phrygian cap) by a man of his own surroundings, a future baron of the Empire. Monsieur Grandet quitted office without regret. He had constructed in the interests of the town certain fine roads which led to his own property; his house and lands, very advantageously assessed, paid moderate taxes; and since the registration of his various estates, the vineyards, thanks to his constant care, had become the “head of the country,”--a local term used to denote those that produced the finest quality of wine. He might have asked for the cross of the Legion of honor. This event occurred in 1806. Monsieur Grandet was then fifty-seven years of age, his wife thirty-six, and an only daughter, the fruit of their legitimate love, was ten years old. Monsieur Grandet, whom Providence no doubt desired to compensate for the loss of his municipal honors, inherited three fortunes in the course of this year,--that of Madame de la Gaudiniere, born de la Bertelliere, the mother of Madame Grandet; that of old Monsieur de la Bertelliere, her grandfather; and, lastly, that of Madame Gentillet, her grandmother on the mother’s side: three inheritances, whose amount was not known to any one. The avarice of the deceased persons was so keen that for a long time they had hoarded their money for the pleasure of secretly looking at it. Old Monsieur de la Bertelliere called an investment an extravagance, and thought he got better interest from the sight of his gold than from the profits of usury. The inhabitants of Saumur consequently estimated his savings according to “the revenues of the sun’s wealth,” as they said. Monsieur Grandet thus obtained that modern title of nobility which our mania for equality can never rub out. He became the most imposing personage in the arrondissement. He worked a hundred acres of vineyard, which in fruitful years yielded seven or eight hundred hogsheads of wine. He owned thirteen farms, an old abbey, whose windows and arches he had walled up for the sake of economy,--a measure which preserved them,--also a hundred and twenty-seven acres of meadow-land, where three thousand poplars, planted in 1793, grew and flourished; and finally, the house in which he lived. Such was his visible estate; as to his other property, only two persons could give even a vague guess at its value: one was Monsieur Cruchot, a notary employed in the usurious investments of Monsieur Grandet; the other was Monsieur des Grassins, the richest banker in Saumur, in whose profits Grandet had a certain covenanted and secret share. Although old Cruchot and Monsieur des Grassins were both gifted with the deep discretion which wealth and trust beget in the provinces, they publicly testified so much respect to Monsieur Grandet that observers estimated the amount of his property by the obsequious attention which they bestowed upon him. In all Saumur there was no one not persuaded that Monsieur Grandet had a private treasure, some hiding-place full of louis, where he nightly took ineffable delight in gazing upon great masses of gold. Avaricious people gathered proof of this when they looked at the eyes of the good man, to which the yellow metal seemed to have conveyed its tints. The glance of a man accustomed to draw enormous interest from his capital acquires, like that of the libertine, the gambler, or the sycophant, certain indefinable habits,--furtive, eager, mysterious movements, which never escape the notice of his co-religionists. This secret language is in a certain way the freemasonry of the passions. Monsieur Grandet inspired the respectful esteem due to one who owed no man anything, who, skilful cooper and experienced wine-grower that he was, guessed with the precision of an astronomer whether he ought to manufacture a thousand puncheons for his vintage, or only five hundred, who never failed in any speculation, and always had casks for sale when casks were worth more than the commodity that filled them, who could store his whole vintage in his cellars and bide his time to put the puncheons on the market at two hundred francs, when the little proprietors had been forced to sell theirs for five louis. His famous vintage of 1811, judiciously stored and slowly disposed of, brought him in more than two hundred and forty thousand francs. Financially speaking, Monsieur Grandet was something between a tiger and a boa-constrictor. He could crouch and lie low, watch his prey a long while, spring upon it, open his jaws, swallow a mass of louis, and then rest tranquilly like a snake in process of digestion, impassible, methodical, and cold. No one saw him pass without a feeling of admiration mingled with respect and fear; had not every man in Saumur felt the rending of those polished steel claws? For this one, Maitre Cruchot had procured the money required for the purchase of a domain, but at eleven per cent. For that one, Monsieur des Grassins discounted bills of exchange, but at a frightful deduction of interest. Few days ever passed that Monsieur Grandet’s name was not mentioned either in the markets or in social conversations at the evening gatherings. To some the fortune of the old wine-grower was an object of patriotic pride. More than one merchant, more than one innkeeper, said to strangers with a certain complacency: “Monsieur, we have two or three millionaire establishments; but as for Monsieur Grandet, he does not himself know how much he is worth.” In 1816 the best reckoners in Saumur estimated the landed property of the worthy man at nearly four millions; but as, on an average, he had made yearly, from 1793 to 1817, a hundred thousand francs out of that property, it was fair to presume that he possessed in actual money a sum nearly equal to the value of his estate. So that when, after a game of boston or an evening discussion on the matter of vines, the talk fell upon Monsieur Grandet, knowing people said: “Le Pere Grandet? le Pere Grandet must have at least five or six millions.” “You are cleverer than I am; I have never been able to find out the amount,” answered Monsieur Cruchot or Monsieur des Grassins, when either chanced to overhear the remark. If some Parisian mentioned Rothschild or Monsieur Lafitte, the people of Saumur asked if he were as rich as Monsieur Grandet. When the Parisian, with a smile, tossed them a disdainful affirmative, they looked at each other and shook their heads with an incredulous air. So large a fortune covered with a golden mantle all the actions of this man. If in early days some peculiarities of his life gave occasion for laughter or ridicule, laughter and ridicule had long since died away. His least important actions had the authority of results repeatedly shown. His speech, his clothing, his gestures, the blinking of his eyes, were law to the country-side, where every one, after studying him as a naturalist studies the result of instinct in the lower animals, had come to understand the deep mute wisdom of his slightest actions. “It will be a hard winter,” said one; “Pere Grandet has put on his fur gloves.” “Pere Grandet is buying quantities of staves; there will be plenty of wine this year.” Monsieur Grandet never bought either bread or meat. His farmers supplied him weekly with a sufficiency of capons, chickens, eggs, butter, and his tithe of wheat. He owned a mill; and the tenant was bound, over and above his rent, to take a certain quantity of grain and return him the flour and bran. La Grande Nanon, his only servant, though she was no longer young, baked the bread of the household herself every Saturday. Monsieur Grandet arranged with kitchen-gardeners who were his tenants to supply him with vegetables. As to fruits, he gathered such quantities that he sold the greater part in the market. His fire-wood was cut from his own hedgerows or taken from the half-rotten old sheds which he built at the corners of his fields, and whose planks the farmers carted into town for him, all cut up, and obligingly stacked in his wood-house, receiving in return his thanks. His only known expenditures were for the consecrated bread, the clothing of his wife and daughter, the hire of their chairs in church, the wages of la Grand Nanon, the tinning of the saucepans, lights, taxes, repairs on his buildings, and the costs of his various industries. He had six hundred acres of woodland, lately purchased, which he induced a neighbor’s keeper to watch, under the promise of an indemnity. After the acquisition of this property he ate game for the first time. Monsieur Grandet’s manners were very simple. He spoke little. He usually expressed his meaning by short sententious phrases uttered in a soft voice. After the Revolution, the epoch at which he first came into notice, the good man stuttered in a wearisome way as soon as he was required to speak at length or to maintain an argument. This stammering, the incoherence of his language, the flux of words in which he drowned his thought, his apparent lack of logic, attributed to defects of education, were in reality assumed, and will be sufficiently explained by certain events in the following history. Four sentences, precise as algebraic formulas, sufficed him usually to grasp and solve all difficulties of life and commerce: “I don’t know; I cannot; I will not; I will see about it.” He never said yes, or no, and never committed himself to writing. If people talked to him he listened coldly, holding his chin in his right hand and resting his right elbow in the back of his left hand, forming in his own mind opinions on all matters, from which he never receded. He reflected long before making any business agreement. When his opponent, after careful conversation, avowed the secret of his own purposes, confident that he had secured his listener’s assent, Grandet answered: “I can decide nothing without consulting my wife.” His wife, whom he had reduced to a state of helpless slavery, was a useful screen to him in business. He went nowhere among friends; he neither gave nor accepted dinners; he made no stir or noise, seeming to economize in everything, even movement. He never disturbed or disarranged the things of other people, out of respect for the rights of property. Nevertheless, in spite of his soft voice, in spite of his circumspect bearing, the language and habits of a coarse nature came to the surface, especially in his own home, where he controlled himself less than elsewhere. Physically, Grandet was a man five feet high, thick-set, square-built, with calves twelve inches in circumference, knotted knee-joints, and broad shoulders; his face was round, tanned, and pitted by the small-pox; his chin was straight, his lips had no curves, his teeth were white; his eyes had that calm, devouring expression which people attribute to the basilisk; his forehead, full of transverse wrinkles, was not without certain significant protuberances; his yellow-grayish hair was said to be silver and gold by certain young people who did not realize the impropriety of making a jest about Monsieur Grandet. His nose, thick at the end, bore a veined wen, which the common people said, not without reason, was full of malice. The whole countenance showed a dangerous cunning, an integrity without warmth, the egotism of a man long used to concentrate every feeling upon the enjoyments of avarice and upon the only human being who was anything whatever to him,--his daughter and sole heiress, Eugenie. Attitude, manners, bearing, everything about him, in short, testified to that belief in himself which the habit of succeeding in all enterprises never fails to give to a man. Thus, though his manners were unctuous and soft outwardly, Monsieur Grandet’s nature was of iron. His dress never varied; and those who saw him to-day saw him such as he had been since 1791. His stout shoes were tied with leathern thongs; he wore, in all weathers, thick woollen stockings, short breeches of coarse maroon cloth with silver buckles, a velvet waistcoat, in alternate stripes of yellow and puce, buttoned squarely, a large maroon coat with wide flaps, a black cravat, and a quaker’s hat. His gloves, thick as those of a gendarme, lasted him twenty months; to preserve them, he always laid them methodically on the brim of his hat in one particular spot. Saumur knew nothing further about this personage. Only six individuals had a right of entrance to Monsieur Grandet’s house. The most important of the first three was a nephew of Monsieur Cruchot. Since his appointment as president of the Civil courts of Saumur this young man had added the name of Bonfons to that of Cruchot. He now signed himself C. de Bonfons. Any litigant so ill-advised as to call him Monsieur Cruchot would soon be made to feel his folly in court. The magistrate protected those who called him Monsieur le president, but he favored with gracious smiles those who addressed him as Monsieur de Bonfons. Monsieur le president was thirty-three years old, and possessed the estate of Bonfons (Boni Fontis), worth seven thousand francs a year; he expected to inherit the property of his uncle the notary and that of another uncle, the Abbe Cruchot, a dignitary of the chapter of Saint-Martin de Tours, both of whom were thought to be very rich. These three Cruchots, backed by a goodly number of cousins, and allied to twenty families in the town, formed a party, like the Medici in Florence; like the Medici, the Cruchots had their Pazzi. Madame des Grassins, mother of a son twenty-three years of age, came assiduously to play cards with Madame Grandet, hoping to marry her dear Adolphe to Mademoiselle Eugenie. Monsieur des Grassins, the banker, vigorously promoted the schemes of his wife by means of secret services constantly rendered to the old miser, and always arrived in time upon the field of battle. The three des Grassins likewise had their adherents, their cousins, their faithful allies. On the Cruchot side the abbe, the Talleyrand of the family, well backed-up by his brother the notary, sharply contested every inch of ground with his female adversary, and tried to obtain the rich heiress for his nephew the president. This secret warfare between the Cruchots and des Grassins, the prize thereof being the hand in marriage of Eugenie Grandet, kept the various social circles of Saumur in violent agitation. Would Mademoiselle Grandet marry Monsieur le president or Monsieur Adolphe des Grassins? To this problem some replied that Monsieur Grandet would never give his daughter to the one or to the other. The old cooper, eaten up with ambition, was looking, they said, for a peer of France, to whom an income of three hundred thousand francs would make all the past, present, and future casks of the Grandets acceptable. Others replied that Monsieur and Madame des Grassins were nobles, and exceedingly rich; that Adolphe was a personable young fellow; and that unless the old man had a nephew of the pope at his beck and call, such a suitable alliance ought to satisfy a man who came from nothing,--a man whom Saumur remembered with an adze in his hand, and who had, moreover, worn the _bonnet rouge_. Certain wise heads called attention to the fact that Monsieur Cruchot de Bonfons had the right of entry to the house at all times, whereas his rival was received only on Sundays. Others, however, maintained that Madame des Grassins was more intimate with the women of the house of Grandet than the Cruchots were, and could put into their minds certain ideas which would lead, sooner or later, to success. To this the former retorted that the Abbe Cruchot was the most insinuating man in the world: pit a woman against a monk, and the struggle was even. “It is diamond cut diamond,” said a Saumur wit. The oldest inhabitants, wiser than their fellows, declared that the Grandets knew better than to let the property go out of the family, and that Mademoiselle Eugenie Grandet of Saumur would be married to the son of Monsieur Grandet of Paris, a wealthy wholesale wine-merchant. To this the Cruchotines and the Grassinists replied: “In the first place, the two brothers have seen each other only twice in thirty years; and next, Monsieur Grandet of Paris has ambitious designs for his son. He is mayor of an arrondissement, a deputy, colonel of the National Guard, judge in the commercial courts; he disowns the Grandets of Saumur, and means to ally himself with some ducal family,--ducal under favor of Napoleon.” In short, was there anything not said of an heiress who was talked of through a circumference of fifty miles, and even in the public conveyances from Angers to Blois, inclusively! At the beginning of 1811, the Cruchotines won a signal advantage over the Grassinists. The estate of Froidfond, remarkable for its park, its mansion, its farms, streams, ponds, forests, and worth about three millions, was put up for sale by the young Marquis de Froidfond, who was obliged to liquidate his possessions. Maitre Cruchot, the president, and the abbe, aided by their adherents, were able to prevent the sale of the estate in little lots. The notary concluded a bargain with the young man for the whole property, payable in gold, persuading him that suits without number would have to be brought against the purchasers of small lots before he could get the money for them; it was better, therefore, to sell the whole to Monsieur Grandet, who was solvent and able to pay for the estate in ready money. The fine marquisate of Froidfond was accordingly conveyed down the gullet of Monsieur Grandet, who, to the great astonishment of Saumur, paid for it, under proper discount, with the usual formalities. This affair echoed from Nantes to Orleans. Monsieur Grandet took advantage of a cart returning by way of Froidfond to go and see his chateau. Having cast a master’s eye over the whole property, he returned to Saumur, satisfied that he had invested his money at five per cent, and seized by the stupendous thought of extending and increasing the marquisate of Froidfond by concentrating all his property there. Then, to fill up his coffers, now nearly empty, he resolved to thin out his woods and his forests, and to sell off the poplars in the meadows"

In [3]:
tr4sh = TextRank4Sentences()
tr4sh.analyze(text)
pprint(tr4sh.get_top_sentences(5), width=1, depth=2)

[1.01637436 1.34690143 1.26918404 1.27827438 1.11597054 1.04863807
 0.99660288 1.09207471 1.01326376 0.52508447 0.91433187 0.81731759
 1.23712541 0.95599142 0.87861621 1.16049065 1.18365472 0.725097
 1.30699967 0.5148984  1.06908645 0.9296075  0.91664791 0.68984557
 0.68509259 0.83464592 1.13641063 1.1357774  0.67982284 0.99427916
 1.06586489 0.92452098 1.10661982 1.16062019 0.77804971 0.65020333
 1.15159649 0.98212787 0.91979003 1.19591508 1.15668326 1.07171942
 1.29863658 0.97956391 0.96186842 1.10873873 1.27229218 1.42846524
 1.12631124 1.0562226  1.41634379 0.95224899 0.78375903 1.28096099
 0.47854632 1.38099191 0.96496629 0.50330111 1.15427252 1.2941163
 0.923606   0.97183939 1.1072169  0.54038633 0.85929352 0.83390475
 1.23336267 1.20693049 1.03339129 0.94736201 1.04394424 1.3695113
 0.9721289  1.27000333 0.95240361 0.82308304 1.16688528 0.71799956
 1.07092832 0.99291329 0.75816673 0.95978407 1.32380518 1.07426077
 0.84611145 0.84816883 0.97402116 0.78682409 0.70764983 0.7601725


**KeyBERT**

In [5]:
from keybert import KeyBERT
kw_model = KeyBERT()
keywords = kw_model.extract_keywords(text)

/Users/taddeocarpinelli/.pyenv/versions/3.10.6/envs/No_Spoil_Zone/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
keywords

[('melancholy', 0.524),
 ('dwelling', 0.4151),
 ('ruins', 0.4013),
 ('picturesque', 0.3826),
 ('cloisters', 0.3679)]

In [20]:
from keybert import KeyBERT
kw_model = KeyBERT()
keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words='english',
                              use_mmr=True, diversity=0.8)

keywords

[('melancholy dwelling', 0.6487),
 ('clean dry narrowness', 0.136),
 ('cent seized stupendous', 0.1313),
 ('joists iron hoops', 0.0325),
 ('signal advantage', -0.0175)]